In [29]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [30]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

Extracting stocks

In [31]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
tickers = pd.read_csv("Tickers.csv", header=None)[0].to_list()
close_prices = pd.DataFrame()

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.info

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock
    if (information["market"] != "ca_market") and (information["market"] != "us_market"):
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information["quoteType"] != "EQUITY":
        print(tick, "not stock")
        continue

    if (information["currency"] != "CAD") and (information["currency"] != "USD"):
        print(tick, "currency wrong")
        continue
    

    temp = yf.download(tick, filter_start, filter_end)

    if temp["Volume"].mean()[tick] < 5000:
        print(tick, "not enough volume")
        continue
    
    close_prices[tick] = temp["Close"]

# Filters out the first and last trading day of each month
idx = close_prices.index.to_series()
date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
monthly_first = []
monthly_last = []
for i in date_series_first:
    monthly_first.append(i)
for i in date_series_last:
    monthly_last.append(i)

for i in range(len(monthly_first)):
    start_ind = close_prices.index.get_loc(monthly_first[i])
    end_ind = close_prices.index.get_loc(monthly_last[i])

    if (end_ind - start_ind + 1) < 18:
        close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$AGN: possibly delisted; no timezone found


AGN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BK: possibly delisted; no price data found  (1d 1927-12-14 -> 2026-11-19)


BK doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$CELG: possibly delisted; no timezone found


CELG doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$MON: possibly delisted; no timezone found


MON doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$RTN: possibly delisted; no timezone found


RTN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [32]:
display(close_prices)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,254.316605,235.580246,129.656540,236.889038,77.725349,220.630005,324.080444,215.199997,49.622257,6.67,...,65.924820,163.093613,208.619995,20.261114,107.826408,174.982285,339.266205,229.894760,78.023796,46.106377
2025-10-02,255.989151,228.041809,129.190277,237.501389,77.725349,222.410004,325.974243,217.429993,49.426430,6.63,...,68.169785,165.405457,211.240005,19.976391,106.874413,176.854431,344.545593,229.679321,79.124153,46.135262
2025-10-03,256.875183,225.487259,130.744537,238.453979,79.072166,219.509995,325.737518,216.300003,49.583088,6.28,...,68.486244,165.728714,224.889999,20.104975,108.913048,174.914383,350.857574,231.912262,80.243011,46.289333
2025-10-06,255.551132,221.901199,129.918839,241.224197,81.024086,220.899994,327.463623,219.729996,49.338310,6.68,...,70.503754,165.180145,229.520004,19.893730,108.951508,176.359711,349.464661,232.509659,80.095062,45.952301
2025-10-07,255.342056,224.446136,129.219421,244.198547,82.029327,221.779999,323.498444,221.820007,49.240398,6.48,...,73.787132,162.084625,225.100006,19.875360,108.605331,171.742416,354.227814,227.074219,79.503273,45.740452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,335.664978,263.502045,100.451622,175.998123,74.820000,249.380005,304.717529,196.800003,56.029999,12.27,...,52.599998,194.259995,205.300003,12.030000,167.495621,269.335388,375.010010,274.070007,90.706924,57.857586
2026-09-25,340.811066,262.726807,100.670280,174.708466,74.169998,249.669998,307.937592,198.070007,56.700001,11.57,...,55.040001,201.970001,201.199997,12.000000,169.511597,276.719360,376.589996,273.790009,92.589050,58.778946
2026-09-28,338.143066,264.654968,100.362175,173.081512,72.779999,246.149994,305.385437,184.389999,55.470001,12.51,...,54.279999,187.479996,204.199997,11.860000,168.429123,276.958191,377.829987,274.100006,92.874825,58.234055


In [33]:
full_data = pd.DataFrame()

for tick in close_prices:
    full_data[tick] = yf.download(tick, period_start, period_end)["Close"]

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data

display(full_data)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,171.301224,132.844559,89.382935,289.540161,56.168285,129.460007,143.918594,187.830002,24.848055,6.02,...,104.262344,73.839996,17.100691,70.040115,146.078217,483.225769,189.679688,128.719498,28.285889,45.243134
2023-10-03,169.970261,132.109726,89.438957,287.633392,54.849430,124.720001,139.653381,188.919998,24.112854,5.98,...,103.408379,70.440002,17.204718,69.378128,143.612396,478.436432,190.746811,127.615570,27.581823,45.229580
2023-10-04,171.212463,132.342728,89.308243,290.601440,55.522888,127.000000,141.174622,186.729996,24.140774,5.86,...,103.089302,72.230003,17.756868,69.195213,144.269943,479.516388,189.333328,127.590683,27.493814,45.243134
2023-10-05,172.444839,132.127670,89.821777,290.855042,56.261829,125.959999,142.199341,186.289993,24.252447,5.20,...,103.042366,71.510002,18.100965,69.997559,141.657959,484.784668,187.807602,127.308464,28.048267,45.247669
2023-10-06,174.988495,132.835587,90.456688,293.231384,56.935280,127.959999,144.355194,187.380005,24.261757,5.15,...,104.055901,73.269997,18.044949,70.359047,143.384079,492.841980,189.511200,128.047211,28.039471,45.256702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,335.664978,263.502045,100.451622,175.998123,74.820000,249.380005,304.717529,196.800003,56.029999,12.27,...,194.259995,205.300003,12.030000,167.495621,269.335388,375.010010,274.070007,90.706924,57.857586,49.906166
2026-09-25,340.811066,262.726807,100.670280,174.708466,74.169998,249.669998,307.937592,198.070007,56.700001,11.57,...,201.970001,201.199997,12.000000,169.511597,276.719360,376.589996,273.790009,92.589050,58.778946,49.906166
2026-09-28,338.143066,264.654968,100.362175,173.081512,72.779999,246.149994,305.385437,184.389999,55.470001,12.51,...,187.479996,204.199997,11.860000,168.429123,276.958191,377.829987,274.100006,92.874825,58.234055,49.906166


In [34]:
half_life = 100

In [51]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return standard_deviation, average

In [60]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

0.00015446020852580053
{'PYPL': np.float64(-117.21740103531118), 'CL': np.float64(-101.15082392764481), 'AIG': np.float64(-99.65039090664405), 'LMT': np.float64(-86.0764435481968), 'ACN': np.float64(-44.41514727815721), 'AXP': np.float64(-44.10168571975226), 'UPS': np.float64(-38.994607357927464), 'ABT': np.float64(-32.8224779726638), 'BA': np.float64(-23.41814531462938), 'PEP': np.float64(-16.759047351834383), 'T.TO': np.float64(-7.562646927209781), 'TD.TO': np.float64(7.155568994814594), 'RY.TO': np.float64(8.57391636016282), 'MRK': np.float64(11.108332530446665), 'BB.TO': np.float64(11.90238824067089), 'AAPL': np.float64(13.39658919666269), 'ABBV': np.float64(14.302960250449042), 'KO': np.float64(14.54188099038884), 'BIIB': np.float64(15.87519821003714), 'PFE': np.float64(16.13762478246188), 'C': np.float64(19.943975678806332), 'TXN': np.float64(19.98388559392544), 'CAT': np.float64(20.11819767909947), 'PM': np.float64(20.444026997010617), 'BMY': np.float64(22.23871242180001), 'LLY'